In [2]:
# Cell 1: Setup - Đảm bảo working directory là segment4
import os
import sys
import logging

# Chuyển working directory về segment4 (quan trọng!)
os.chdir('/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, '/home/hieu0606sunny/price2026wsl/tech2ai/segment4')

print(f"Working directory: {os.getcwd()}")

# Setup logging
logging.basicConfig(level=logging.INFO)
root = logging.getLogger()
root.setLevel(logging.INFO)

from dotenv import load_dotenv
load_dotenv(override=True)

# Check BRAVE_API_KEY
brave_key = os.getenv("BRAVE_API_KEY")
print(f"BRAVE_API_KEY: {'✅ Found' if brave_key else '❌ Not found'}")

Working directory: /home/hieu0606sunny/price2026wsl/tech2ai/segment4
BRAVE_API_KEY: ✅ Found


In [1]:
# Cell 2: Import dependencies
import asyncio
from typing import List

from openai import OpenAI
from pydantic import BaseModel, Field
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

print("✅ Imports successful!")

✅ Imports successful!


In [3]:
# Cell 3: Define SearchResults schema cho Amazon
# Giống như BestBuy nhưng cho Amazon

class SearchResults(BaseModel):
    """URLs found from Brave Search"""
    product_urls: List[str] = Field(description="List of Amazon product URLs")

print("✅ SearchResults schema defined!")
print(f"   Schema: {SearchResults.schema()}")

✅ SearchResults schema defined!
   Schema: {'description': 'URLs found from Brave Search', 'properties': {'product_urls': {'description': 'List of Amazon product URLs', 'items': {'type': 'string'}, 'title': 'Product Urls', 'type': 'array'}}, 'required': ['product_urls'], 'title': 'SearchResults', 'type': 'object'}


/tmp/ipykernel_20728/1984836345.py:9: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  print(f"   Schema: {SearchResults.schema()}")


In [4]:
# Cell 4: Define AmazonSearchAgent
# Tương tự BestBuySearchAgent nhưng điều chỉnh prompt cho Amazon

class AmazonSearchAgent:
    """
    Agent that uses Brave MCP Server to search for Amazon product URLs.
    
    This agent:
    1. Takes a keyword (e.g., "laptop")
    2. Uses Brave Search with site:amazon.com filter
    3. Returns a list of Amazon product URLs (format: /dp/ASIN or /gp/product/ASIN)
    """
    
    name = "Amazon Search Agent"
    MODEL = "gpt-5-nano"
    
    # INSTRUCTIONS cho việc tìm kiếm Amazon
    INSTRUCTIONS = """You are a web search agent. Your job is to find product URLs on Amazon.

STEPS:
1. Use brave_web_search with the given search query
2. Extract product URLs from results
3. Return ONLY URLs that are Amazon product pages

VALID Amazon product URL patterns (based on actual results):
- https://www.amazon.com/PRODUCT-NAME/dp/XXXXXXXXXX
- https://www.amazon.com/*/dp/XXXXXXXXXX
- Examples:
  - https://www.amazon.com/amazon-fire-tv-43-inch-4-series-4k-smart-tv/dp/B0CZ9WV2ZX
  - https://www.amazon.com/Apple-2025-MacBook-13-inch-Laptop/dp/B0DZD9S5GC
  - https://www.amazon.com/ASUS-Gaming-Laptop-Nebula-Display/dp/B0DW1X5YCQ

IMPORTANT:
- The ASIN code (after /dp/) is 10 characters (letters and numbers)
- Do NOT return search pages (amazon.com/s?k=...)
- Do NOT return category or brand pages
- Do NOT return Amazon homepage
- Return as many product URLs as you can find
"""
    
    def __init__(self):
        """Initialize the search agent."""
        logging.info(f"[{self.name}] Initializing...")
        self.brave_api_key = os.getenv("BRAVE_API_KEY")
        if not self.brave_api_key:
            raise ValueError("BRAVE_API_KEY not found in environment variables")
        logging.info(f"[{self.name}] Ready!")
    
    def get_brave_params(self) -> dict:
        """Get Brave MCP server parameters."""
        return {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-brave-search"],
            "env": {"BRAVE_API_KEY": self.brave_api_key}
        }
    
    async def _search_async(self, keyword: str, max_urls: int = 15) -> List[str]:
        """
        Async implementation of search.
        
        Args:
            keyword: Product keyword to search (e.g., "laptop")
            max_urls: Maximum number of URLs to return
            
        Returns:
            List of Amazon product URLs
        """
        logging.info(f"[{self.name}] Searching for: {keyword}")
        
        async with MCPServerStdio(
            params=self.get_brave_params(),
            client_session_timeout_seconds=60
        ) as brave_server:
            
            # Sử dụng trace để quan sát
            with trace(workflow_name="Amazon Search", group_id=keyword):
                search_agent = Agent(
                    name="AmazonSearchAgent",
                    instructions=self.INSTRUCTIONS,
                    model=self.MODEL,
                    mcp_servers=[brave_server],
                    output_type=SearchResults
                )
                
                result = await Runner.run(
                    search_agent,
                    f"Search for {keyword} on Amazon. Find product pages with /dp/ in the URL.",
                    max_turns=30
                )
                
                urls = result.final_output.product_urls[:max_urls]
                logging.info(f"[{self.name}] Found {len(urls)} product URLs")
                return urls
    
    def search(self, keyword: str, max_urls: int = 15) -> List[str]:
        """
        Search for Amazon products using Brave Search.
        
        Args:
            keyword: Product keyword to search (e.g., "laptop", "headphones")
            max_urls: Maximum number of URLs to return (default: 15)
            
        Returns:
            List of Amazon product URLs
        """
        try:
            loop = asyncio.get_running_loop()
        except RuntimeError:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            return loop.run_until_complete(self._search_async(keyword, max_urls))
        else:
            import nest_asyncio
            nest_asyncio.apply()
            return loop.run_until_complete(self._search_async(keyword, max_urls))


print("✅ AmazonSearchAgent class defined!")

✅ AmazonSearchAgent class defined!


In [ ]:
# Cell 5: Test AmazonSearchAgent với keyword "laptop"
# 🔍 Trace sẽ được ghi lại tại: https://platform.openai.com/traces

# Khởi tạo agent
search_agent = AmazonSearchAgent()

# Tìm kiếm sản phẩm
keyword = "Smart TV"
print(f"🔍 Searching for '{keyword}' on Amazon...")
print(f"   Using Brave MCP + GPT-5-nano")
print(f"   Trace: Check https://platform.openai.com/traces for details\n")

urls = search_agent.search(keyword, max_urls=10)

print(f"\n{'='*70}")
print(f"✅ Found {len(urls)} product URLs:")
print(f"{'='*70}\n")

for i, url in enumerate(urls, 1):
    # Highlight nếu URL có /dp/ (đúng format)
    is_valid = "/dp/" in url or "/gp/product/" in url
    status = "✓" if is_valid else "⚠️"
    print(f"{i}. {status} {url}")

INFO:root:[Amazon Search Agent] Initializing...
INFO:root:[Amazon Search Agent] Ready!
INFO:root:[Amazon Search Agent] Searching for: Smart TV


🔍 Searching for 'Smart TV' on Amazon...
   Using Brave MCP + GPT-5-nano
   Trace: Check https://platform.openai.com/traces for details



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:root:[Amazon Search Agent] Found 10 product URLs



✅ Found 10 product URLs:

1. ✓ https://www.amazon.com/amazon-fire-tv-40-inch-2-series-hd-smart-tv/dp/B0CJCYMBZJ
2. ✓ https://www.amazon.com/amazon-fire-tv-32-inch-2-series-hd-smart-tv/dp/B0CJDSNN4T
3. ✓ https://www.amazon.com/amazon-fire-tv-55-inch-4-series-4k-smart-tv/dp/B0CZBS9GKV
4. ✓ https://www.amazon.com/amazon-fire-tv-50-inch-4-series-4k-smart-tv/dp/B0CZBLZYY5
5. ✓ https://www.amazon.com/amazon-fire-tv-43-inch-4-series-4k-smart-tv/dp/B0CZ9WV2ZX
6. ✓ https://www.amazon.com/introducing-amazon-fire-tv-40-inch-2-series-hd-smart-tv/dp/B09N719G17
7. ✓ https://www.amazon.com/amazon-fire-tv-50-inch-omni-series-4k-smart-tv/dp/B08T6F8YBH
8. ✓ https://www.amazon.com/amazon-fire-tv-55-inch-omni-mini-led-series-smart-tv/dp/B0C7SRHGXF
9. ✓ https://www.amazon.com/amazon-fire-tv-55-inch-omni-series-4k-smart-tv/dp/B08P3QVFMK
10. ✓ https://www.amazon.com/amazon-fire-tv-65-inch-omni-qled-series-smart-tv/dp/B0BJMGB9RN


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


In [38]:
# Cell 6: Amazon sale checker với việc set US Zip Code trước
from playwright.async_api import async_playwright
from typing import Tuple, List
import asyncio
import logging

logger = logging.getLogger(__name__)

# US Zip Code - California (same as your Windows setting)
US_ZIP_CODE = "96150"


async def set_amazon_us_location(page) -> bool:
    """
    Set Amazon delivery location to US by entering zip code.
    This needs to be done once before checking URLs.
    
    Args:
        page: Playwright page object
        
    Returns:
        True if successfully set location
    """
    try:
        print(f"📍 Setting Amazon location to US (Zip: {US_ZIP_CODE})...")
        
        # Go to Amazon homepage first
        await page.goto("https://www.amazon.com", timeout=30000, wait_until="domcontentloaded")
        await page.wait_for_timeout(2000)
        
        # Click on "Deliver to" location selector
        # This element is usually at top left, with id "nav-global-location-popover-link"
        location_btn = page.locator("#nav-global-location-popover-link")
        
        if await location_btn.count() > 0:
            await location_btn.click()
            await page.wait_for_timeout(1500)
            
            # Find zip code input field
            zip_input = page.locator('input[data-action="GLUXPostalInputAction"]')
            
            if await zip_input.count() > 0:
                # Clear and enter zip code
                await zip_input.fill(US_ZIP_CODE)
                await page.wait_for_timeout(500)
                
                # Click Apply button
                apply_btn = page.locator('input[aria-labelledby="GLUXZipUpdate-announce"]')
                if await apply_btn.count() > 0:
                    await apply_btn.click()
                    await page.wait_for_timeout(2000)
                    print(f"✅ Location set to US (Zip: {US_ZIP_CODE})")
                    return True
                else:
                    # Try alternative apply button
                    apply_btn2 = page.locator('span[data-action="GLUXPostalUpdateAction"] input')
                    if await apply_btn2.count() > 0:
                        await apply_btn2.click()
                        await page.wait_for_timeout(2000)
                        print(f"✅ Location set to US (Zip: {US_ZIP_CODE})")
                        return True
            else:
                # Maybe need to click "Change" first if already has location
                change_btn = page.locator('a[id="GLUXChangePostalCodeLink"]')
                if await change_btn.count() > 0:
                    await change_btn.click()
                    await page.wait_for_timeout(1000)
                    # Retry entering zip code
                    zip_input = page.locator('input[data-action="GLUXPostalInputAction"]')
                    if await zip_input.count() > 0:
                        await zip_input.fill(US_ZIP_CODE)
                        apply_btn = page.locator('input[aria-labelledby="GLUXZipUpdate-announce"]')
                        if await apply_btn.count() > 0:
                            await apply_btn.click()
                            await page.wait_for_timeout(2000)
                            print(f"✅ Location set to US (Zip: {US_ZIP_CODE})")
                            return True
        
        print("⚠️ Could not find location elements, but continuing...")
        return False
        
    except Exception as e:
        print(f"⚠️ Error setting location: {e}")
        return False


async def is_on_sale_amazon_playwright(url: str, page) -> Tuple[bool, dict]:
    """
    Check if Amazon product is on sale using Playwright.
    """
    try:
        await page.goto(url, timeout=30000, wait_until="domcontentloaded")
        await page.wait_for_timeout(2500)
        
        price_info = {}
        
        # Check sale indicator 1: savingsPercentage
        savings_elem = page.locator("span.savingsPercentage")
        has_savings = await savings_elem.count() > 0
        
        if has_savings:
            savings_text = await savings_elem.first.text_content()
            price_info["savings_pct"] = savings_text.strip()
        
        # Check sale indicator 2: basisPrice
        basis_elem = page.locator("span.basisPrice")
        has_basis = await basis_elem.count() > 0
        
        # Check sale indicator 3: data-a-strike
        strike_elem = page.locator('[data-a-strike="true"]')
        has_strike = await strike_elem.count() > 0
        
        # Get current price
        price_elem = page.locator("span.priceToPay")
        if await price_elem.count() > 0:
            price_text = await price_elem.first.text_content()
            price_info["sale_price"] = price_text.strip()
        
        is_sale = has_savings or has_basis or has_strike
        return is_sale, price_info
        
    except Exception as e:
        logger.warning(f"Error checking {url}: {e}")
        return False, {}


async def filter_amazon_sale_urls_playwright(urls: List[str]) -> List[Tuple[str, dict]]:
    """
    Filter Amazon URLs to keep only products on sale.
    Sets US location first, then checks each URL.
    """
    sale_items = []
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,
            args=['--disable-blink-features=AutomationControlled', '--no-sandbox']
        )
        
        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
            viewport={'width': 1920, 'height': 1080},
            locale="en-US",
            timezone_id="America/New_York",
        )
        
        page = await context.new_page()
        
        # ===== STEP 1: Set US Location =====
        await set_amazon_us_location(page)
        
        # ===== STEP 2: Check each URL =====
        print(f"\n🔍 Checking {len(urls)} URLs for sale items...")
        print("="*60)
        
        for i, url in enumerate(urls, 1):
            is_sale, price_info = await is_on_sale_amazon_playwright(url, page)
            
            if is_sale:
                sale_items.append((url, price_info))
                savings = price_info.get("savings_pct", "N/A")
                price = price_info.get("sale_price", "N/A")
                print(f"[{i}/{len(urls)}] 🏷️ SALE | {savings} | {price}")
            else:
                price = price_info.get("sale_price", "No price found")
                print(f"[{i}/{len(urls)}] ⏭️ Skip | Price: {price}")
            
            await page.wait_for_timeout(500)
        
        await browser.close()
    
    print("="*60)
    print(f"✅ Found {len(sale_items)}/{len(urls)} products on sale")
    
    return sale_items


print("✅ Amazon sale checker with US ZIP code defined!")
print(f"📍 Will set location to ZIP: {US_ZIP_CODE} before checking")

✅ Amazon sale checker with US ZIP code defined!
📍 Will set location to ZIP: 96150 before checking


In [40]:
urls

['https://www.amazon.com/amazon-fire-tv-55-inch-4-series-4k-smart-tv/dp/B0CZBS9GKV',
 'https://www.amazon.com/amazon-fire-tv-32-inch-2-series-hd-smart-tv/dp/B0CJDSNN4T',
 'https://www.amazon.com/amazon-fire-tv-50-inch-4-series-4k-smart-tv/dp/B0CZBLZYY5',
 'https://www.amazon.com/amazon-fire-tv-40-inch-2-series-hd-smart-tv/dp/B0CJCYMBZJ',
 'https://www.amazon.com/amazon-fire-tv-55-inch-omni-mini-led-series-smart-tv/dp/B0C7SRHGXF',
 'https://www.amazon.com/amazon-fire-tv-50-inch-4-series-4k-smart-tv/dp/B08SVZ775L',
 'https://www.amazon.com/amazon-fire-tv-43-inch-4-series-4k-smart-tv/dp/B0B3HG269B',
 'https://www.amazon.com/introducing-amazon-fire-tv-40-inch-2-series-hd-smart-tv/dp/B09N719G17',
 'https://www.amazon.com/amazon-fire-tv-55-inch-omni-series-4k-smart-tv/dp/B08P3QVFMK',
 'https://www.amazon.com/introducing-amazon-fire-tv-32-inch-2-series-hd-smart-tv/dp/B09N6F9NV3']

In [41]:
# Cell 7: Test - xem browser set location thành công không

test_urls = [
    'https://www.amazon.com/Apple-2025-MacBook-13-inch-Laptop/dp/B0DZD9S5GC?th=1',  # SALE
    'https://www.amazon.com/amazon-fire-tv-55-inch-4-series-4k-smart-tv/dp/B0CZ9WV2ZX?th=1',  # NOT SALE
]

print("🚀 Testing with 2 known URLs...")
print("📍 Browser will set ZIP code to 96150 first\n")

sale_items = await filter_amazon_sale_urls_playwright(urls)

print(f"\n📋 Sale items found ({len(sale_items)}):")
for url, price_info in sale_items:
    print(f"   URL: {url[:60]}...")
    print(f"   Price info: {price_info}")

🚀 Testing with 2 known URLs...
📍 Browser will set ZIP code to 96150 first

📍 Setting Amazon location to US (Zip: 96150)...
✅ Location set to US (Zip: 96150)

🔍 Checking 10 URLs for sale items...
[1/10] ⏭️ Skip | Price: $185.93
[2/10] ⏭️ Skip | Price: $149.99
[3/10] ⏭️ Skip | Price: $338.26
[4/10] ⏭️ Skip | Price: $249.99
[5/10] 🏷️ SALE | -12% | $719.99
[6/10] ⏭️ Skip | Price: No price found
[7/10] ⏭️ Skip | Price: $185.93
[8/10] ⏭️ Skip | Price: $249.99
[9/10] ⏭️ Skip | Price: $407.99
[10/10] ⏭️ Skip | Price: $149.99
✅ Found 1/10 products on sale

📋 Sale items found (1):
   URL: https://www.amazon.com/amazon-fire-tv-55-inch-omni-mini-led-...
   Price info: {'savings_pct': '-12%', 'sale_price': '$719.99'}
